# Od grafu do przewidywania gola: GAT + GRU

Ten notebook bierze grafy z `04_graphs` i uczy sie z nich przewidywać, czy w ciągu **10 minut** padnie gol.
Kolejność jest taka sama jak w `04_graphs` — każdy krok osobno, z kontrolą, że liczby się zgadzają:

1. wczytanie zrzutów grafu,
2. etykiety „gol w ciągu 10 minut",
3. sekwencje `T = 10` zrzutów,
4. podział po meczach (bez wycieku),
5. składanie batcha i warianty kanałów krawędzi,
6. model: GAT (uwaga na sąsiadach) → pooling → GRU → dwie głowy,
7. trening trzech wariantów,
8. baseline MLP bez grafu (czy graf w ogóle pomaga),
9. ewaluacja: AUC, PR-AUC, Brier, kalibracja, bootstrap po meczach.

Cel jest **dwugłowy**: osobno prawdopodobieństwo gola gospodarzy i gości.


## Krok 0 — wczytanie zrzutów

Plik `graphs.npz` to wynik `scripts/build_graphs.py`: jeden wiersz na minutę każdego meczu. Trzy tablice
plus `meta` z numerem meczu, połową i minutą. Dogrywki (periody 3 i 4) wyrzucamy — etykieta „gol w ciągu
10 minut" ma sens tylko w regularnym czasie gry.


In [1]:
# zrzuty grafu: 303 mecze, jeden zrzut na minute
from pathlib import Path

import numpy as np
import pandas as pd

OUT = Path.cwd().parent / "output"

d = np.load(OUT / "graphs.npz")
X, A, A_loss, meta = d["X"], d["A"], d["A_loss"], d["meta"]

# dogrywki (periody 3 i 4) wyrzucamy - etykieta "10 minut" ma sens tylko w regularnym czasie
regularne = np.isin(meta[:, 1], [1, 2])
X, A, A_loss, meta = X[regularne], A[regularne], A_loss[regularne], meta[regularne]

podania = A.copy()
podania[:, 22, :] = 0
podania[:, :, 22] = 0            # krawedzie pilka-gracz nie sa podaniami

print("zrzuty:", X.shape[0], "| X:", X.shape, "| A:", A.shape, "| A_loss:", A_loss.shape)
print("mecze:", len(np.unique(meta[:, 0])), "| polowy:", sorted(np.unique(meta[:, 1]).tolist()))
_, ile_na_mecz = np.unique(meta[:, 0], return_counts=True)
print("zrzutow na mecz: min %d | mediana %d | max %d" % (
    ile_na_mecz.min(), int(np.median(ile_na_mecz)), ile_na_mecz.max()))
print("krawedzi podan na zrzut: %.1f" % (podania > 0).sum(axis=(1, 2)).mean())
print("krawedzi strat na zrzut: %.1f" % (A_loss > 0).sum(axis=(1, 2)).mean())
print("krawedzi pilka-gracz na zrzut: %.1f" % (
    (A[:, 22, :] > 0).sum(axis=1) + (A[:, :, 22] > 0).sum(axis=1)).mean())


zrzuty: 30276 | X: (30276, 23, 11) | A: (30276, 23, 23) | A_loss: (30276, 23, 23)
mecze: 303 | polowy: [1, 2]
zrzutow na mecz: min 93 | mediana 99 | max 116
krawedzi podan na zrzut: 28.4
krawedzi strat na zrzut: 8.8
krawedzi pilka-gracz na zrzut: 1.9


## Krok 1 — etykieta: gol w ciągu 10 minut

Dla każdego zrzutu i każdej drużyny pytamy: **czy w przedziale `(t, t + 10 min]` padł jej gol**.
Przedział jest otwarty z lewej, więc gol zdobyty dokładnie w minucie `t` jeszcze się nie liczy (to już przeszłość,
model ma go przewidzieć, a nie przepisać wynik).

Gole bierzemy z tabeli zdarzeń, nie z grafu. **Samobój zaliczamy drużynie przeciwnej** — inaczej etykieta
mówiłaby „gospodarze strzelili", gdy w rzeczywistości stracili gola.


In [2]:
DELTA = 10        # horyzont celu w minutach

ev = pd.read_csv(OUT / "combined" / "combined_events.csv", usecols=[
    "match_id", "team_side", "is_goal", "is_own_goal", "expanded_minute", "second", "period_value"])
ev = ev[ev["period_value"].isin([1, 2])].copy()
ev["czas"] = ev["expanded_minute"] + ev["second"].fillna(0) / 60

gole = ev[ev["is_goal"]].copy()
gole["strona"] = np.where(gole["is_own_goal"],
                          np.where(gole["team_side"] == "home", "away", "home"),
                          gole["team_side"])
czasy = {s: gole[gole["strona"] == s].groupby("match_id")["czas"].apply(np.sort).to_dict()
         for s in ("home", "away")}

y_home = np.zeros(len(meta), dtype=np.float32)
y_away = np.zeros(len(meta), dtype=np.float32)
for strona, y in (("home", y_home), ("away", y_away)):
    for mid, tab in czasy[strona].items():
        w = meta[:, 0] == mid
        t = meta[w, 2]
        y[w] = (np.searchsorted(tab, t + DELTA, side="right")
                > np.searchsorted(tab, t, side="right")).astype(np.float32)

print("goli w regularnym czasie: %d (samobojow %d)" % (len(gole), int(gole["is_own_goal"].sum())))
print("gol gospodarzy w ciagu %d min: %.2f%% zrzutow" % (DELTA, 100 * y_home.mean()))
print("gol gosci     w ciagu %d min: %.2f%% zrzutow" % (DELTA, 100 * y_away.mean()))

# przyklad: jak etykieta wynika z konkretnych goli
przyklad = int(meta[0, 0])
wiersze = np.where(meta[:, 0] == przyklad)[0][:5]
print("\nprzyklad: mecz %d, gole gospodarzy w minutach %s" % (
    przyklad, np.round(czasy["home"].get(przyklad, []), 1)))
for i in wiersze:
    t = meta[i, 2]
    w_oknie = [g for g in czasy["home"].get(przyklad, []) if t < g <= t + DELTA]
    print("  minuta %2d -> y_home=%d (gole w (%2d, %2d]: %s)" % (
        t, y_home[i], t, t + DELTA, ", ".join("%.1f" % g for g in w_oknie) or "brak"))


goli w regularnym czasie: 906 (samobojow 36)
gol gospodarzy w ciagu 10 min: 15.01% zrzutow
gol gosci     w ciagu 10 min: 11.99% zrzutow

przyklad: mecz 1729190, gole gospodarzy w minutach [25.4 31.8]
  minuta  1 -> y_home=0 (gole w ( 1, 11]: brak)
  minuta  2 -> y_home=0 (gole w ( 2, 12]: brak)
  minuta  3 -> y_home=0 (gole w ( 3, 13]: brak)
  minuta  4 -> y_home=0 (gole w ( 4, 14]: brak)
  minuta  5 -> y_home=0 (gole w ( 5, 15]: brak)


## Krok 2 — sekwencja 10 zrzutów

Sieć nie patrzy na pojedynczy zrzut, tylko na **10 kolejnych minut** (GRU czyta je w kolejności).
Żeby taki ciąg miał sens, muszą być spełnione trzy warunki:

1. **jest historia** — zrzut ma 10 wcześniejszych (odpadają pierwsze 9 minut każdej połowy),
2. **ta sama połowa** — sekwencja nie skleja 45:00 z 46:00 (przerwa to nie gra),
3. **pełne okno etykiety** — `t + 10` musi mieścić się w połowie, inaczej etykieta byłaby niepełna
   (gol w 47. minucie nie „należy" do okna kończącego się na 45+).


In [3]:
T = 10            # dlugosc sekwencji w zrzutach (minuty)

granica = pd.DataFrame({"mid": meta[:, 0], "polowa": meta[:, 1], "t": meta[:, 2]})
maks_t = granica.groupby(["mid", "polowa"])["t"].transform("max").to_numpy()

n = len(meta)
zrodlo = np.arange(n) - (T - 1)
ma_historie = zrodlo >= 0
zrodlo_bezp = np.clip(zrodlo, 0, None)
ta_sama_polowa = (meta[zrodlo_bezp, 1] == meta[:, 1]) & (meta[zrodlo_bezp, 0] == meta[:, 0])
pelne_okno = meta[:, 2] + DELTA <= maks_t

zostaje = ma_historie & ta_sama_polowa & pelne_okno
idx = np.arange(n)[zostaje]

print("zrzutow: %d" % n)
print("  odrzucone: brak historii (pierwsze %d minut polowy) %5d" % (T - 1, int((~ma_historie).sum())))
print("  odrzucone: przeskok polowy                        %5d" % int((ma_historie & ~ta_sama_polowa).sum()))
print("  odrzucone: okno etykiety wychodzi z polowy        %5d" % int((ma_historie & ta_sama_polowa & ~pelne_okno).sum()))
print("probek: %d (%.0f%%) | srednio na mecz: %.0f" % (
    len(idx), 100 * len(idx) / n, len(idx) / len(np.unique(meta[idx, 0]))))

pierwsza = int(idx[0])
print("\nprzyklad: probka %d to zrzuty %d..%d (mecz %d, polowa %d, minuty %d-%d)" % (
    pierwsza, pierwsza - T + 1, pierwsza, meta[pierwsza, 0], meta[pierwsza, 1],
    meta[pierwsza - T + 1, 2], meta[pierwsza, 2]))


zrzutow: 30276
  odrzucone: brak historii (pierwsze 9 minut polowy)     9
  odrzucone: przeskok polowy                         5445
  odrzucone: okno etykiety wychodzi z polowy         6060
probek: 18762 (62%) | srednio na mecz: 62

przyklad: probka 9 to zrzuty 0..9 (mecz 1729190, polowa 1, minuty 1-10)


## Krok 3 — podział po meczach

Dzielimy **całe mecze**, nie próbki: 70% trening, 15% walidacja, 15% test. Inaczej sąsiednie minuty tego samego
meczu wpadłyby jednocześnie do treningu i do testu — model „rozpoznałby" mecz, a nie nauczył się gry.


In [4]:
TEST, VAL, SEED = 0.15, 0.15, 42

mecze_prob = meta[idx, 0]
rng = np.random.default_rng(SEED)
unikalne = np.unique(mecze_prob)
rng.shuffle(unikalne)
n_test = int(len(unikalne) * TEST)
n_val = int(len(unikalne) * VAL)
test_m = set(unikalne[:n_test].tolist())
val_m = set(unikalne[n_test:n_test + n_val].tolist())

mask_test = np.isin(mecze_prob, list(test_m))
mask_val = np.isin(mecze_prob, list(val_m))
mask_train = ~(mask_test | mask_val)

probki = {"train": idx[mask_train], "val": idx[mask_val], "test": idx[mask_test]}
etykiety = {"train": (y_home[probki["train"]], y_away[probki["train"]]),
            "val": (y_home[probki["val"]], y_away[probki["val"]]),
            "test": (y_home[probki["test"]], y_away[probki["test"]])}

for nazwa, maska in (("train", mask_train), ("val", mask_val), ("test", mask_test)):
    y = np.concatenate([etykiety[nazwa][0], etykiety[nazwa][1]])
    print("%-5s %5d probek | %3d meczow | goli w etykiecie %.1f%%" % (
        nazwa, maska.sum(), len(np.unique(mecze_prob[maska])), 100 * y.mean()))


train 13212 probek | 213 meczow | goli w etykiecie 14.7%
val    2749 probek |  45 meczow | goli w etykiecie 13.1%
test   2801 probek |  45 meczow | goli w etykiecie 13.2%


## Krok 4 — składanie batcha i warianty kanałów

Sekwencja próbki to po prostu `T` kolejnych zrzutów, więc batch powstaje jednym wycięciem z tablicy —
bez pętli po próbkach i bez trzymania w pamięci wszystkich sekwencji naraz.

Krawędzie mogą wejść na trzy sposoby:

| wariant | co wchodzi | ile kanałów |
|---|---|---|
| **A** | tylko podania | 1 |
| **B** | podania **i** straty w jednej masce | 1 |
| **C** | podania i straty rozdzielone, z wyuczonym biasem na typ | 2 |

W kanale podań siedzą też dwie krawędzie `piłka ↔ zawodnik przy piłce` (dokładane w `04_graphs`) — dlatego
wariant C ma „tylko" dwa kanały, a nie trzy jak w wersji eventowej.


In [5]:
def paczka(wybor, k=T):
    """(probki, k, 23, 11) cech + dwa kanaly krawedzi (probki, k, 23, 23)."""
    kroki = np.asarray(wybor, dtype=np.int64)[:, None] + np.arange(-k + 1, 1)[None, :]
    return X[kroki], A[kroki], A_loss[kroki]


UKLADY = {
    "A: tylko podania": dict(kanaly=("podania",), typy=0),
    "B: podania + straty": dict(kanaly=("podania", "straty"), typy=0),
    "C: podania + straty osobno": dict(kanaly=("podania", "straty"), typy=2),
}


def kanaly(podania, straty, uklad):
    """Jeden kanal (maski zlaczone) albo dwa kanaly z wlasnym biasem."""
    if uklad["kanaly"] == ("podania",):
        return [podania]
    if uklad["typy"] == 0:
        return [podania + straty]
    return [podania, straty]


# kontrola: paczka musi byc tym samym, co reczne wyciecie z tablicy
wybor = probki["train"][:4]
Xb, Pb, Lb = paczka(wybor)
i0 = int(probki["train"][0])
print("paczka:", Xb.shape, Pb.shape, Lb.shape)
print("  zgodne z recznym wycinkiem X:      ", bool(np.array_equal(Xb[0], X[i0 - T + 1:i0 + 1])))
print("  zgodne z recznym wycinkiem A:      ", bool(np.array_equal(Pb[0], A[i0 - T + 1:i0 + 1])))
print("  zgodne z recznym wycinkiem A_loss: ", bool(np.array_equal(Lb[0], A_loss[i0 - T + 1:i0 + 1])))

print("\nkanaly i gestosc krawedzi:")
for nazwa, uklad in UKLADY.items():
    lista = kanaly(Pb, Lb, uklad)
    print("  %-28s kanalow %d | krawedzi na zrzut %s" % (
        nazwa, len(lista), [round(float(k.sum()) / (len(wybor) * T), 1) for k in lista]))


paczka: (4, 10, 23, 11) (4, 10, 23, 23) (4, 10, 23, 23)
  zgodne z recznym wycinkiem X:       True
  zgodne z recznym wycinkiem A:       True
  zgodne z recznym wycinkiem A_loss:  True

kanaly i gestosc krawedzi:
  A: tylko podania             kanalow 1 | krawedzi na zrzut [11.7]
  B: podania + straty          kanalow 1 | krawedzi na zrzut [17.7]
  C: podania + straty osobno   kanalow 2 | krawedzi na zrzut [11.7, 5.9]


## Krok 5 — model

Dwie warstwy GAT i jedna warstwa GRU:

- **GAT** — każdy węzeł waży tylko sąsiadów z maski (`a > 0`); wagi uwagi liczone są z cech węzłów
  (`e_i + e_j`, bez budowania macierzy `N × N × D` — inaczej brakuje pamięci). Przy wariancie C do logitu uwagi
  dochodzi **jedna wyuczona liczba na kanał** (podanie / strata).
- **pooling** — średnia po 23 węzłach daje jeden wektor na zrzut,
- **GRU** — czyta 10 takich wektorów w kolejności,
- **dwie głowy** — osobny wynik dla goli gospodarzy i gości.

Do tego wspólny `trenuj()`, z którego korzysta i GAT, i baseline MLP — żeby porównanie było uczciwe
(ta sama pętla, ten sam optimizer, ten sam early stop po walidacyjnym AUC).


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

URZ = "cuda" if torch.cuda.is_available() else "cpu"
BATCH, EPOKI, CIERPLIWOSC, LR = 128, 12, 3, 1e-3
print("urzadzenie:", URZ, "|", torch.cuda.get_device_name(0) if URZ == "cuda" else "-")


class GAT(nn.Module):
    """Uwaga na sasiadach z listy kanalow; opcjonalnie bias na typ krawedzi."""

    def __init__(self, wej, wyj, glowy=4, dropout=0.2, typy=0):
        super().__init__()
        self.W = nn.Linear(wej, wyj * glowy, bias=False)
        self.uwaga = nn.Parameter(torch.empty(glowy, 2 * wyj))
        nn.init.xavier_uniform_(self.uwaga)
        self.glowy, self.wyj = glowy, wyj
        self.drop = nn.Dropout(dropout)
        self.biasy = nn.ParameterList([nn.Parameter(torch.zeros(())) for _ in range(typy)])

    def forward(self, h, kanaly):
        B, N, _ = h.shape
        Wh = self.W(h).view(B, N, self.glowy, self.wyj)
        e_i = torch.einsum("bnhd,hd->bnh", Wh, self.uwaga[:, :self.wyj]).unsqueeze(2)
        e_j = torch.einsum("bnhd,hd->bnh", Wh, self.uwaga[:, self.wyj:]).unsqueeze(1)
        e = F.leaky_relu(e_i + e_j, 0.2)

        maska = torch.zeros_like(kanaly[0])
        for i, kanal in enumerate(kanaly):
            maska = maska + kanal
            if i < len(self.biasy):
                e = e + self.biasy[i] * kanal.unsqueeze(-1)

        e = e.masked_fill(~(maska > 0).unsqueeze(-1), -1e9)
        alfa = self.drop(torch.softmax(e, dim=2))
        out = torch.einsum("bijh,bjhd->bihd", alfa, Wh)
        return out.reshape(B, N, self.glowy * self.wyj)


class GAT_GRU(nn.Module):
    def __init__(self, wej=11, ukryte=32, glowy=4, gru=48, dropout=0.2, typy=0):
        super().__init__()
        self.gat1 = GAT(wej, ukryte, glowy, dropout, typy)
        self.gat2 = GAT(ukryte * glowy, ukryte, 1, dropout, typy)
        self.gru = nn.GRU(ukryte, gru, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.glowa_home = nn.Linear(gru, 1)
        self.glowa_away = nn.Linear(gru, 1)

    def forward(self, x, kanaly):
        B, Tz, N, C = x.shape
        plaskie = [k.reshape(B * Tz, N, N) for k in kanaly]
        h = self.gat2(F.elu(self.gat1(x.reshape(B * Tz, N, C), plaskie)), plaskie)
        g = h.mean(1).reshape(B, Tz, -1)
        out, _ = self.gru(g)
        ostatni = self.drop(out[:, -1, :])
        return self.glowa_home(ostatni).squeeze(-1), self.glowa_away(ostatni).squeeze(-1)


def metryki(p, y):
    """AUC-ROC, PR-AUC (average precision) i Brier.

    Rowne oceny traktujemy jako JEDEN prog. Inaczej model ze stala predykcja dostaje
    PR zawyzony o przypadkowa kolejnosc wierszy (u nas najpierw gospodarze, potem goscie),
    a poprawna wartoscia dla stalej jest po prostu czestosc goli.
    """
    from scipy.stats import rankdata

    p, y = np.asarray(p, float), np.asarray(y, float)
    npos = int((y == 1).sum())
    r = rankdata(p)
    auc_ = (r[y == 1].sum() - npos * (npos + 1) / 2) / (npos * (len(y) - npos))

    o = np.argsort(-p, kind="mergesort")
    ps, ys = p[o], y[o]
    granice = np.r_[0, np.nonzero(np.diff(ps))[0] + 1, len(ps)]
    tp = np.cumsum(ys)
    prec = tp / np.maximum(tp + np.cumsum(1 - ys), 1)
    rec = tp / npos
    ap = 0.0
    for k in range(1, len(granice)):
        poprzednia = rec[granice[k - 1] - 1] if k > 1 else 0.0
        ap += prec[granice[k] - 1] * (rec[granice[k] - 1] - poprzednia)
    return auc_, float(ap), float(np.mean((p - y) ** 2))


def na_urzadzenie(wybor, uklad):
    Xb, Pb, Lb = paczka(wybor)
    return (torch.tensor(Xb, device=URZ),
            [torch.tensor(k, device=URZ) for k in kanaly(Pb, Lb, uklad)])


def jakosc(model, wybor, uklad, ety, porcja=512):
    """Metryki na nieskalibrowanych prawdopodobienstwach + surowe logity (do kalibracji)."""
    model.eval()
    lo_h, lo_a = [], []
    with torch.no_grad():
        for start in range(0, len(wybor), porcja):
            x, kan = na_urzadzenie(wybor[start:start + porcja], uklad)
            oh, oa = model(x, kan)
            lo_h.append(oh.cpu().numpy())
            lo_a.append(oa.cpu().numpy())
    logity = np.concatenate(lo_h + lo_a)
    y = np.concatenate([ety[0], ety[1]])
    return metryki(1 / (1 + np.exp(-logity)), y), logity, y


def trenuj(model, uklad, epoki=EPOKI, cierpliwosc=CIERPLIWOSC, opis=""):
    """Wspolna petla treningu: Adam, pos_weight, early stop po walidacyjnym AUC."""
    optymalizator = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    waga = float((1 - etykiety["train"][0].mean()) / etykiety["train"][0].mean())
    strata = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(waga, device=URZ))
    gen = np.random.default_rng(SEED)
    naj, stan, bez_poprawy = 0.0, None, 0

    for epoka in range(1, epoki + 1):
        model.train()
        kolejnosc = gen.permutation(len(probki["train"]))
        suma = 0.0
        for start in range(0, len(kolejnosc), BATCH):
            b = kolejnosc[start:start + BATCH]
            x, kan = na_urzadzenie(probki["train"][b], uklad)
            oh, oa = model(x, kan)
            l = strata(oh, torch.tensor(etykiety["train"][0][b], device=URZ))
            l = l + strata(oa, torch.tensor(etykiety["train"][1][b], device=URZ))
            optymalizator.zero_grad()
            l.backward()
            optymalizator.step()
            suma += l.item() * len(b)

        (auc_va, _, br_va), _, _ = jakosc(model, probki["val"], uklad, etykiety["val"])
        print("  %-26s epoka %2d | strata %.4f | val AUC %.3f | val Brier %.4f"
              % (opis, epoka, suma / len(kolejnosc), auc_va, br_va), flush=True)

        if auc_va > naj:
            naj, bez_poprawy = auc_va, 0
            stan = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            bez_poprawy += 1
            if bez_poprawy >= cierpliwosc:
                print("  early stop", flush=True)
                break

    model.load_state_dict(stan)
    return naj, model


model = GAT_GRU(wej=11, typy=3).to(URZ)
print("parametrow GAT+GRU:", sum(p.numel() for p in model.parameters()))


urzadzenie: cuda | NVIDIA GeForce RTX 3060 Laptop GPU
parametrow GAT+GRU: 17736


## Krok 6 — trening trzech wariantów

Ten sam model, ta sama pętla, ten sam ziarno losowości — zmienia się tylko to, co wchodzi na wejście.
Różnica w wynikach pochodzi wyłącznie od kanałów krawędzi.


In [7]:
modele, auc_val = {}, {}
for nazwa, uklad in UKLADY.items():
    torch.manual_seed(SEED)
    model = GAT_GRU(wej=11, typy=uklad["typy"]).to(URZ)
    auc_val[nazwa], modele[nazwa] = trenuj(model, uklad, opis=nazwa)
    if uklad["typy"]:
        print("  biasy kanalow (warstwa 1):", [round(v.item(), 3) for v in modele[nazwa].gat1.biasy])
        print("  biasy kanalow (warstwa 2):", [round(v.item(), 3) for v in modele[nazwa].gat2.biasy])
    torch.save(modele[nazwa].state_dict(), OUT / ("gat_gru_%s.pt" % nazwa[0]))


  A: tylko podania           epoka  1 | strata 2.2514 | val AUC 0.586 | val Brier 0.2263
  A: tylko podania           epoka  2 | strata 2.2391 | val AUC 0.597 | val Brier 0.2377
  A: tylko podania           epoka  3 | strata 2.2356 | val AUC 0.597 | val Brier 0.2361
  A: tylko podania           epoka  4 | strata 2.2275 | val AUC 0.606 | val Brier 0.2246
  A: tylko podania           epoka  5 | strata 2.2258 | val AUC 0.598 | val Brier 0.2409
  A: tylko podania           epoka  6 | strata 2.2229 | val AUC 0.607 | val Brier 0.2313
  A: tylko podania           epoka  7 | strata 2.2191 | val AUC 0.598 | val Brier 0.2306
  A: tylko podania           epoka  8 | strata 2.2225 | val AUC 0.608 | val Brier 0.2273
  A: tylko podania           epoka  9 | strata 2.2193 | val AUC 0.601 | val Brier 0.2449
  A: tylko podania           epoka 10 | strata 2.2162 | val AUC 0.591 | val Brier 0.2397
  A: tylko podania           epoka 11 | strata 2.2147 | val AUC 0.607 | val Brier 0.2393
  early stop
  B: pod

## Krok 7 — baseline bez grafu

Żeby powiedzieć „graf pomaga", trzeba mieć z czym porównać. Baseline dostaje **te same dane, ale bez struktury**:
uśrednia cechy po 23 węzłach i wrzuca 10 × 11 liczb do zwykłej sieci MLP. Nie widzi, kto z kim grał —
widzi tylko „jak wygląda przeciętny zawodnik w tych 10 minutach".

Trenujemy go **tą samą funkcją `trenuj()`**, więc różnica w wynikach bierze się z obecności grafu, nie z przepisów.


In [8]:
class MLP(nn.Module):
    """Baseline: srednia cech po wezlach, splaszczona po czasie. Krawedzi nie widzi."""

    def __init__(self, wej):
        super().__init__()
        self.siec = nn.Sequential(nn.Linear(wej, 64), nn.ReLU(), nn.Dropout(0.2),
                                  nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 2))

    def forward(self, x, kanaly):
        out = self.siec(x.mean(2).reshape(len(x), -1))
        return out[:, 0], out[:, 1]


torch.manual_seed(SEED)
mlp = MLP(T * X.shape[2]).to(URZ)
print("parametrow MLP:", sum(p.numel() for p in mlp.parameters()),
      "| wejscie:", T * X.shape[2], "liczb")
auc_val["MLP"], mlp = trenuj(mlp, UKLADY["A: tylko podania"], epoki=8, opis="MLP bez grafu")


parametrow MLP: 9250 | wejscie: 110 liczb
  MLP bez grafu              epoka  1 | strata 2.2544 | val AUC 0.556 | val Brier 0.2341
  MLP bez grafu              epoka  2 | strata 2.2520 | val AUC 0.567 | val Brier 0.2401
  MLP bez grafu              epoka  3 | strata 2.2517 | val AUC 0.572 | val Brier 0.2342
  MLP bez grafu              epoka  4 | strata 2.2514 | val AUC 0.569 | val Brier 0.2288
  MLP bez grafu              epoka  5 | strata 2.2506 | val AUC 0.583 | val Brier 0.2380
  MLP bez grafu              epoka  6 | strata 2.2498 | val AUC 0.581 | val Brier 0.2356
  MLP bez grafu              epoka  7 | strata 2.2498 | val AUC 0.586 | val Brier 0.2368
  MLP bez grafu              epoka  8 | strata 2.2495 | val AUC 0.585 | val Brier 0.2386


## Krok 8 — ewaluacja

Trzy rzeczy, bez których ten wynik byłby nieuczciwy:

1. **Kalibracja Platta** — sieć trenowana z `pos_weight` wypuszcza zawyżone prawdopodobieństwa. Dopasowujemy
   `p = sigmoid(a · logit + b)` na walidacji i dopiero tak skalibrowaną predykcję mierzymy na teście.
   AUC i PR-AUC się nie zmieniają (są rangowe), ale Brier tak.
2. **Brier i PR-AUC obok AUC** — przy 13% pozytywów sama trafność nic nie mówi, a AUC nie widzi kalibracji.
3. **Bootstrap po meczach** — 45 meczów testowych to mało. Losujemy mecze ze zwracaniem i patrzymy,
   jak często różnica między modelami jest dodatnia; to jedyny sposób, żeby odróżnić efekt od szumu.


In [9]:
from scipy.optimize import minimize
from scipy.special import expit

KRATKO = {"A: tylko podania": "A", "B: podania + straty": "B",
          "C: podania + straty osobno": "C", "MLP": "MLP"}


def kalibruj(logit_val, y_val):
    """p = sigmoid(a * logit + b), dopasowane na walidacji."""
    def nll(par):
        p = expit(par[0] * logit_val + par[1])
        return -np.mean(y_val * np.log(p + 1e-9) + (1 - y_val) * np.log(1 - p + 1e-9))

    return minimize(nll, x0=[1.0, 0.0], method="Nelder-Mead").x


def ocena(model, uklad, nazwa):
    _, lv, yv = jakosc(model, probki["val"], uklad, etykiety["val"])
    a, b = kalibruj(lv, yv)
    _, lt, yt = jakosc(model, probki["test"], uklad, etykiety["test"])
    p = expit(a * lt + b)
    auc_, ap, br = metryki(p, yt)
    print("  %-28s %6.3f %6.3f %7.4f   (kalibracja a=%.2f, b=%+.2f)" % (nazwa, auc_, ap, br, a, b))
    return p, yt


y_test = np.concatenate([etykiety["test"][0], etykiety["test"][1]])
print("=== zbior testowy (%d probek, %d meczow) ===" % (
    len(y_test), len(np.unique(mecze_prob[mask_test]))))
print("  %-28s %6s %6s %7s" % ("model", "AUC", "PR", "Brier"))

p_baza = np.full(len(y_test), etykiety["train"][0].mean())
print("  %-28s %6.3f %6.3f %7.4f" % (("baseline (stala)",) + metryki(p_baza, y_test)))

predykcje = {}
predykcje["MLP"], _ = ocena(mlp, UKLADY["A: tylko podania"], "MLP (cechy bez grafu)")
for nazwa, uklad in UKLADY.items():
    predykcje[nazwa], _ = ocena(modele[nazwa], uklad, nazwa)

print()
print("czestosc goli w tescie: %.2f%%" % (100 * y_test.mean()), end="")
for nazwa in predykcje:
    print(" | %s: %.3f" % (KRATKO[nazwa], predykcje[nazwa].mean()), end="")
print()

mid_test = mecze_prob[mask_test]
mid_prob = np.concatenate([mid_test, mid_test])


def bootstrap(jeden, drugi, losowan=300):
    rng_b = np.random.default_rng(0)
    roznice = []
    for _ in range(losowan):
        losowe = rng_b.choice(np.unique(mid_prob), size=len(np.unique(mid_prob)), replace=True)
        wybor = np.concatenate([np.where(mid_prob == m)[0] for m in losowe])
        if y_test[wybor].sum() < 5:
            continue
        roznice.append(metryki(predykcje[jeden][wybor], y_test[wybor])[0]
                       - metryki(predykcje[drugi][wybor], y_test[wybor])[0])
    return np.array(roznice)


print()
print("bootstrap po meczach (300 losowan, %d meczow testowych):" % len(np.unique(mid_prob)))
for jeden, drugi in (("C: podania + straty osobno", "MLP"),
                     ("C: podania + straty osobno", "A: tylko podania"),
                     ("C: podania + straty osobno", "B: podania + straty")):
    roznice = bootstrap(jeden, drugi)
    print("  AUC %-3s - %-3s: %+.3f, 95%% CI [%+.3f, %+.3f], lepszy w %.0f%% losowan" % (
        KRATKO[jeden], KRATKO[drugi], roznice.mean(),
        np.percentile(roznice, 2.5), np.percentile(roznice, 97.5),
        100 * (roznice > 0).mean()))


=== zbior testowy (5602 probek, 45 meczow) ===
  model                           AUC     PR   Brier
  baseline (stala)              0.500  0.132  0.1151
  MLP (cechy bez grafu)         0.565  0.169  0.1145   (kalibracja a=1.62, b=-1.80)
  A: tylko podania              0.614  0.183  0.1123   (kalibracja a=1.55, b=-1.76)
  B: podania + straty           0.626  0.201  0.1116   (kalibracja a=1.52, b=-1.77)
  C: podania + straty osobno    0.623  0.198  0.1117   (kalibracja a=1.52, b=-1.77)

czestosc goli w tescie: 13.17% | MLP: 0.131 | A: 0.129 | B: 0.131 | C: 0.130

bootstrap po meczach (300 losowan, 45 meczow testowych):
  AUC C   - MLP: +0.058, 95% CI [+0.012, +0.116], lepszy w 99% losowan
  AUC C   - A  : +0.009, 95% CI [-0.004, +0.020], lepszy w 92% losowan
  AUC C   - B  : -0.002, 95% CI [-0.005, +0.001], lepszy w 6% losowan


# Wnioski

Model porządkuje momenty meczu lepiej niż losowo: AUC-ROC 0,626 (w 62,6% par trafnie wskazuje moment poprzedzający gola), PR-AUC 0,201 przy bazie 0,132. W praktyce: przy wskazaniu 10% najbardziej prawdopodobnych momentów gol pada w ciągu 10 minut w 28% z nich (baza 13%, lift 2,1×), co pokrywa 21% wszystkich goli. Brier 0,1116 vs 0,1151 dla predykcji stałej — poprawa mała, bo 87% próbek to brak gola, więc metryka jest zdominowana przez przypadki łatwe.
